# 실패 로그와 평가 대상

Python 3.12 새 커널에서 Ch12 검색 설정 36행과 Ch15 제공 답변 읽기부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 평가 사례는 입력 질문, 시스템 출력, 기대 근거 또는 기대 답, 실행 설정을 한 행에 묶습니다. 실패 로그에서 정답 문서가 낮은 순위였던 질문을 찾고, 그 질문에 실제 생성한 답을 연결해야 검색 실패와 생성 실패를 나눠 볼 수 있습니다.

**다른 상황에 적용.** 고객지원 봇에서는 ‘환불 규정을 못 찾음’과 ‘규정은 찾았지만 답에 잘못 옮김’을 별도 사례로 저장할 수 있습니다. 같은 최종 오답이라도 수정할 단계가 다르므로 검색 후보와 최종 답을 모두 남깁니다.

**비교와 경계.** 로그에 남았다는 이유만으로 평가 정답이 되지는 않습니다. 사용자의 불만, 낮은 검색 순위, 긴 지연은 검토할 사례를 고르는 신호입니다. 기대 답과 허용 범위는 사람이 확인한 평가 데이터에 따로 기록해야 합니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** 이 셀은 앞 셀에서 만든 값을 입력으로 읽어 위에서 아래 순서로 처리하고, 왼쪽 변수에 다음 셀이 사용할 결과를 저장합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## Ch12 검색 설정 36행과 Ch15 제공 답변 읽기

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# 프로젝트 루트를 찾아 Ch12 검색 설정 36행과 Ch15 제공 답변 기록을 읽습니다.
from pathlib import Path
import json
root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'labs/part3/ch12-retrieval-tuning').is_dir())
ch12 = root / 'labs/part3/ch12-retrieval-tuning/runs'
rows = [json.loads(x) for x in (ch12 / 'advanced-continuity-experiment-log.jsonl').read_text().splitlines()]
cases = json.loads((root / 'labs/part4/ch15-evaluation-advanced/data/evaluation-cases.json').read_text())
print('검색 설정 행:', len(rows))
print('36행의 생성 응답 필드:', sum('response' in row for row in rows))
print('새로 생성한 답변:', len(cases['cases']))

검색 설정 행: 36
36행의 생성 응답 필드: 0
새로 생성한 답변: 3


## 정답 문서가 1위가 아닌 시험 질문 찾기

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 시험 결과에서 정답 문서가 1위가 아닌 검색 순위 사례만 고릅니다.
test = json.loads((ch12 / 'advanced-continuity-test-result.json').read_text())
low = [r for r in test['results'] if r['ndcg_at_5'] < 1]
for row in low:
    first = row['top_results'][0]['doc_id']
    print(row['qid'], '첫 문서=', first, 'gold=', row['gold_doc_ids'],
          'Recall@5=', row['recall_at_5'], 'nDCG@5=', round(row['ndcg_at_5'], 4))

Q006 첫 문서= E02 gold= ['E01'] Recall@5= 1.0 nDCG@5= 0.6309
Q090 첫 문서= E02 gold= ['H01'] Recall@5= 1.0 nDCG@5= 0.6309


## 실제 생성 답변의 모델과 프롬프트 버전 확인

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 준비 단계에서 실제 생성해 저장한 답변의 모델과 프롬프트 버전을 확인합니다.
for case in cases['cases']:
    print(case['qid'], case['answer_model'], case['answer_prompt_version'])
    print(case['response'][:100])
assert len(rows) == 36 and len(low) == 2
assert all('response' in case for case in cases['cases'])

Q002 gpt-5.4-mini rag-answer-v1.0.0
도서는 한 번에 7권까지 빌릴 수 있고, 대출 기간은 14일입니다. [E01]
Q006 gpt-5.4-mini rag-answer-v1.0.0
회원은 주차장을 최초 2시간 무료로 이용할 수 있고, 그 이후에는 30분마다 500원의 요금이 부과됩니다. [E01]
Q090 gpt-5.4-mini rag-answer-v1.0.0
제공된 근거에는 “입회인”이 누구인지에 대한 내용이 별도로 기재되어 있지 않아 알 수 없습니다. [H01]
